# Vegetation Trend Monitoring: A Real Time Series Project — Obuasi Mining Belt, Ghana

**Multi-date NDVI trend and anomaly detection, with a clip-first processing pipeline.**

This notebook exists because a naive two-date "before vs. after" comparison
is fragile — pick an unlucky pair of dates (both cloudy, both in the same
season, both coincidentally similar) and you can conclude "no change"
when real change is happening, or the reverse. A genuine time series —
several dates spanning multiple years, fit with an actual trend line —
is far more robust, and is what `pygeofetch.processor.TimeSeriesAnalyzer`
is built for.

## What makes this the "powerful" version

| | Two-date comparison | This notebook |
|---|---|---|
| Dates used | 2 | 6, spanning 2018-2024 |
| Change detection | Simple subtraction | Per-pixel linear trend (vectorized least-squares) |
| Statistical grounding | None | Anomaly detection (z-score vs. a baseline period) |
| Processing efficiency | Full scene processed, then analyzed | **Clipped to the exact AOI boundary before any scaling/masking/index computation** — smaller arrays, faster, and never processes pixels outside the area you actually care about |
| Output | A single difference map | Trend map, zonal time series, anomaly map, all georeferenced |

## The clip-first pipeline

```mermaid
graph LR
    A[Download scene] --> B[Extract raw bands]
    B --> C[Clip to AOI boundary]
    C --> D[Radiometric scaling]
    D --> E[Cloud masking]
    E --> F[Stack across dates]
    F --> G[Trend / Zonal series / Anomaly]
```

Clipping happens **immediately after extraction, before scaling or cloud
masking** — every downstream step then operates only on the pixels that
actually matter, not the full scene.


In [1]:
from pathlib import Path

import numpy as np

from pygeofetch import PyGeoFetch
from pygeofetch.models.download_task import DownloadOptions
from pygeofetch.models.search_query import SearchQuery
from pygeofetch.processing.preprocessor import Preprocessor
from pygeofetch.processor import LandsatExtractor, TimeSeriesAnalyzer
from pygeofetch.viz import MapViewer, Plotter

print("Modules loaded: PyGeoFetch, LandsatExtractor, SpectralIndex, TimeSeriesAnalyzer,")
print("                Preprocessor (for clipping), Plotter, MapViewer")


Modules loaded: PyGeoFetch, LandsatExtractor, SpectralIndex, TimeSeriesAnalyzer,
                Preprocessor (for clipping), Plotter, MapViewer


## 1. Study Area — Obuasi Municipal District (real boundary, not a rectangle)

Same verified polygon used throughout this project — centroid 6.200°N,
1.683°W, area 96.4 km² (matches the published 96.42 km² figure to
within 0.1%). See the earlier notebooks for the full sourcing note on
why this is a hand-verified approximation rather than an authoritative
shapefile pull.


In [2]:
import json

obuasi_boundary_coords = [
    [-1.7427, 6.2277], [-1.7210, 6.2381], [-1.6946, 6.2413],
    [-1.6665, 6.2341], [-1.6465, 6.2221], [-1.6328, 6.2036],
    [-1.6360, 6.1820], [-1.6545, 6.1635], [-1.6809, 6.1531],
    [-1.7082, 6.1555], [-1.7322, 6.1691], [-1.7467, 6.1900],
    [-1.7531, 6.2092], [-1.7427, 6.2277],
]
obuasi_geometry = {"type": "Polygon", "coordinates": [obuasi_boundary_coords]}

obuasi_lons = [c[0] for c in obuasi_boundary_coords]
obuasi_lats = [c[1] for c in obuasi_boundary_coords]
aoi_extent = (min(obuasi_lons), max(obuasi_lons), min(obuasi_lats), max(obuasi_lats))

output_dir = Path("./data/obuasi_timeseries")
output_dir.mkdir(parents=True, exist_ok=True)

boundary_path = output_dir / "obuasi_boundary.geojson"
boundary_path.write_text(json.dumps({
    "type": "FeatureCollection",
    "features": [{"type": "Feature", "properties": {"name": "Obuasi Municipal District"}, "geometry": obuasi_geometry}],
}))

print("Study area: Obuasi Municipal District, Ashanti Region, Ghana")
print(f"Vertices: {len(obuasi_boundary_coords)} (irregular polygon)")
print(f"Extent: {aoi_extent}")


Study area: Obuasi Municipal District, Ashanti Region, Ghana
Vertices: 14 (irregular polygon)
Extent: (-1.7531, -1.6328, 6.1531, 6.2413)


## 2. Search — six dates spanning 2018-2024

Spreading dates across multiple years, rather than a single before/after
pair, is what gives `TimeSeriesAnalyzer.trend()` genuine data to fit a
slope from — a real least-squares fit through 6 points is far more
robust than a difference between 2.


In [3]:
client = PyGeoFetch(log_level="INFO")

# Replace with your own credentials — see notebook 12 for the full
# authentication walkthrough and M2M access prerequisites.
USGS_USERNAME = "SamuelYamforo"
USGS_APP_TOKEN = "pZR!9XSr85145X8fMa7tJkEbs_!esM3Z4iV_EY5FHiuEO1oD@R1eM7PRUOTXNLn1"
try:
    client.add_credentials("usgs", username=USGS_USERNAME, api_key=USGS_APP_TOKEN)
except Exception as exc:
    print(f"Could not register credentials: {exc}")

# One search window per year, spread across the dry season (Nov-Feb) where
# possible for more consistent cloud cover and comparable vegetation phenology
date_windows = [
    ("2018-11-01", "2019-02-28"),
    ("2019-11-01", "2020-02-28"),
    ("2020-11-01", "2021-02-28"),
    ("2021-11-01", "2022-02-28"),
    ("2022-11-01", "2023-02-28"),
    ("2023-11-01", "2024-02-28"),
]

scenes = {}
for start, end in date_windows:
    query = SearchQuery(
        geometry=obuasi_geometry, start_date=start, end_date=end,
        satellites=["Landsat-8", "Landsat-9"], cloud_cover_max=20, max_results=5,
    )
    try:
        results = client.search(query, providers=["usgs"])
    except Exception as exc:
        print(f"Search failed for {start}..{end}: {exc}")
        results = []

    if results:
        best = min(results, key=lambda r: r.cloud_cover or 100)
        scenes[str(best.datetime.date())] = best
        print(f"  {start[:4]}: {best.id}  ({best.datetime.date()}, {best.cloud_cover:.0f}% cloud)")
    else:
        print(f"  {start[:4]}: no scenes found")

have_real_data = len(scenes) >= 3  # need at least 3 dates for a meaningful trend
print(f"\n{len(scenes)} dates found — {'sufficient' if have_real_data else 'insufficient'} for time series analysis")
if not have_real_data:
    print("Falls back to a synthetic demonstration in Section 7.")


11:17:05 INFO [      engine] PyGeoFetch ready
11:17:05 INFO [authenticator] Credentials saved for provider 'usgs'
┌ SEARCH PARAMETERS ───────────────────────────────────────────────────────┐
│ Providers  : usgs                                                        │
│ BBox       : —                                                           │
│ Date range : 2018-11-01  →  2019-02-28                                   │
│ Cloud max  : 20.0%                                                       │
│ Product    : any                                                         │
└──────────────────────────────────────────────────────────────────────────┘
11:17:08 INFO [        usgs] Authenticated with USGS M2M API as 'SamuelYamforo'
  ✓  usgs                             5 scenes   67.9s
┌────────────────────────────────────────────┬────────────┬────────────────┬────────┬─────────┬──────────────┬─────────────┬───────┬───────┬──────────────────────┐
│                  SCENE ID                  │  

AttributeError: 'NoneType' object has no attribute 'date'

## 3. Download all scenes


In [ ]:
download_results = {}

if have_real_data:
    options = DownloadOptions(parallel=1, resume=True, on_failure="skip")
    for date_str, scene in scenes.items():
        scene_dir = output_dir / date_str
        results_dl = client.download([scene], destination=scene_dir, options=options)
        r = results_dl[0]
        status = "OK" if r.success else f"FAILED: {r.error}"
        print(f"  {date_str}: {status}")
        if r.success:
            download_results[date_str] = r

    have_real_data = len(download_results) >= 3


## 4. Clip-first processing — the key difference from a naive pipeline

For each date: extract the raw bands from the downloaded bundle, **clip
each raw band to the Obuasi boundary immediately**, then apply radiometric
scaling and cloud masking to the clipped (small) arrays — not the full
scene. Every later step in this notebook only ever touches the pixels
inside the actual study area.


In [ ]:
extractor = LandsatExtractor()  # mask_clouds handled separately below, on clipped data
pp = Preprocessor()

def process_scene_clip_first(source, work_dir, aoi_geometry, bands=("red", "nir")):
    """
    Extract -> clip to AOI -> scale -> cloud-mask, in that order.

    Returns {band_name: scaled+masked array} plus the profile of the
    clipped grid, or None if the scene couldn't be processed.
    """
    work_dir = Path(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)

    # Resolve to either individual downloaded files or a bundle to extract —
    # reuses LandsatExtractor's own delivery-style detection so this stays
    # consistent with how process_scene() handles the same two cases.
    individual = extractor._resolve_individual_files(source)
    if individual is not None:
        extracted = individual
        sensor_hint = next(iter(extracted.keys()), "")
    else:
        bundle_path = extractor._resolve_path(source)
        if bundle_path is None:
            return None
        extracted = extractor.extract_bundle(bundle_path, work_dir / "extracted")
        sensor_hint = bundle_path.name

    sensor = extractor._detect_sensor(sensor_hint)
    band_map = {"OLI": {"red": "SR_B4", "nir": "SR_B5"}, "TM": {"red": "SR_B3", "nir": "SR_B4"}}[sensor]

    # Clip each raw band (and QA_PIXEL, for cloud masking) to the AOI FIRST
    clipped_paths = {}
    for band_name in list(bands) + ["qa_pixel"]:
        suffix = band_map.get(band_name) if band_name in band_map else "QA_PIXEL"
        raw_path = extractor.find_band(extracted, suffix)
        if raw_path is None:
            continue
        clip_result = pp.clip(raw_path, geometry=aoi_geometry, output=str(work_dir / f"{band_name}_clipped.tif"))
        if clip_result.success:
            clipped_paths[band_name] = clip_result.output_path

    if not all(b in clipped_paths for b in bands):
        return None

    # THEN scale (on the clipped, much smaller arrays)
    band_arrays = {}
    profile = None
    for band_name in bands:
        arr, prof = extractor.load_scaled_band(clipped_paths[band_name])
        if profile is None:
            profile = prof
        band_arrays[band_name.upper()] = arr

    # THEN cloud-mask (also on the clipped grid)
    if "qa_pixel" in clipped_paths:
        mask = extractor.cloud_mask(clipped_paths["qa_pixel"], shape=next(iter(band_arrays.values())).shape)
        for k in band_arrays:
            band_arrays[k] = np.where(mask, np.nan, band_arrays[k])

    return band_arrays, profile, sensor


date_bands = {}
clip_profile = None
for date_str, result in download_results.items():
    processed = process_scene_clip_first(result, output_dir / date_str, obuasi_geometry)
    if processed is None:
        print(f"  {date_str}: could not process (missing bands)")
        continue
    band_arrays, profile, sensor = processed
    clip_profile = clip_profile or profile
    date_bands[date_str] = {"RED": None, "NIR": None}  # placeholder — real arrays written to disk below

    # TimeSeriesAnalyzer.build_index_stack() reads bands from FILES, so
    # write the clipped+scaled arrays back out rather than passing arrays
    # directly — this also gives you real, inspectable intermediate files.
    for band_key, arr in band_arrays.items():
        band_name = "RED" if band_key == "RED" else "NIR"
        out_path = output_dir / date_str / f"{band_name.lower()}_scaled_clipped.tif"
        prof = profile.copy()
        prof.update(dtype="float32", count=1)
        import rasterio
        with rasterio.open(out_path, "w", **prof) as dst:
            dst.write(arr.astype(np.float32)[np.newaxis, :, :])
        date_bands[date_str][band_name] = out_path

    print(f"  {date_str}: clipped + scaled + masked ({sensor}, grid shape {next(iter(band_arrays.values())).shape})")

have_real_data = len(date_bands) >= 3


## 5. Build the NDVI time series and compute a real trend

`TimeSeriesAnalyzer` computes NDVI for every date automatically and
stacks the results — this is the piece that turns "N separate rasters"
into an actual analyzable time series.


In [ ]:
ts = TimeSeriesAnalyzer(index="NDVI")

if have_real_data:
    stack = ts.build_index_stack(date_bands)
    print(stack)

    trend = ts.trend(stack)
    print(f"\nTrend range: [{np.nanmin(trend):.4f}, {np.nanmax(trend):.4f}] NDVI/year")
    print(f"Mean trend: {np.nanmean(trend):.4f} NDVI/year")

    declining_pct = 100 * np.nanmean(trend < -0.02)
    print(f"Area with clear declining trend (< -0.02 NDVI/year): {declining_pct:.1f}%")


## 6. Zonal time series and anomaly detection

The zonal series gives the actual NDVI trajectory for the whole district
— the direct "did vegetation here go up or down over these 6 years"
answer, in one number per date. The anomaly map flags where the most
recent date departs most from the established baseline.


In [ ]:
if have_real_data:
    zonal_df = ts.zonal_timeseries(stack, boundary_path, zone_id_field="name")
    print(zonal_df.to_string(index=False))

    # First 2 dates as the baseline "normal" period, most recent as target
    baseline_dates = stack.dates[:2]
    anomaly = ts.anomaly(stack, baseline=baseline_dates, target=stack.dates[-1])
    print(f"\nAnomaly (z-score) range: [{np.nanmin(anomaly):.2f}, {np.nanmax(anomaly):.2f}]")
    print(f"Area with strong negative anomaly (z < -2): {100*np.nanmean(anomaly < -2):.1f}%")


## 7. Visualize — trend map, classification, and the actual time series plot


In [ ]:
pl = Plotter(figsize=(12, 8))

if have_real_data:
    pl.plot_raster(
        trend, title="Obuasi — NDVI Trend, 2018-2024 (dry-season composites)",
        colormap="RdBu", vmin=-0.05, vmax=0.05, extent=aoi_extent,
        colorbar_label="NDVI / year",
        output=str(output_dir / "trend_map.png"),
    )


In [ ]:
TREND_CLASS_LABELS = {0: "Strong decline", 1: "Moderate decline", 2: "Stable", 3: "Increasing"}
TREND_CLASS_COLORS = {0: "#d73027", 1: "#fc8d59", 2: "#91cf60", 3: "#1a9850"}

def classify_trend(trend_arr):
    codes = np.full(trend_arr.shape, 2, dtype=np.int16)
    codes[trend_arr < -0.03] = 0
    codes[(trend_arr >= -0.03) & (trend_arr < -0.01)] = 1
    codes[trend_arr > 0.01] = 3
    codes[np.isnan(trend_arr)] = 2
    return codes

if have_real_data:
    trend_classified = classify_trend(trend)
    pl.plot_classification(
        trend_classified, class_labels=TREND_CLASS_LABELS, class_colors=TREND_CLASS_COLORS,
        title="Obuasi — Vegetation Trend Classification (2018-2024)",
        extent=aoi_extent, output=str(output_dir / "trend_classification.png"),
    )


In [ ]:
if have_real_data:
    series = ts.zone_series(stack, boundary_path, zone_id="Obuasi Municipal District", zone_id_field="name")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 5))
    dates_dt = list(series.keys())
    values = list(series.values())
    ax.plot(dates_dt, values, marker="o", linewidth=2, color="#2ecc71")
    ax.set_title("Obuasi Municipal District — Mean NDVI, 2018-2024")
    ax.set_ylabel("NDVI")
    ax.grid(alpha=0.3)
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(output_dir / "district_ndvi_timeseries.png", dpi=120)
    plt.show()
    print(f"District-wide NDVI trajectory: {values}")


## 8. Interactive map


In [ ]:
if have_real_data:
    from pygeofetch.processing.base import _safe_write_band
    trend_tif = output_dir / "trend.tif"
    _safe_write_band(trend, clip_profile, trend_tif)

    try:
        mv = MapViewer(center=(6.198, -1.693), zoom=12)
        mv.add_vector(str(boundary_path), layer_name="Obuasi Municipal District",
                      style={"color": "#2c3e50", "weight": 2, "fillOpacity": 0.0})
        mv.add_raster(str(trend_tif), colormap="RdBu", vmin=-0.05, vmax=0.05,
                      layer_name="NDVI Trend", opacity=0.75)
        mv.add_basemap("SATELLITE")
        map_path = output_dir / "interactive_trend_map.html"
        mv.save(map_path)
        print(f"Interactive map saved: {map_path}")
    except ImportError as exc:
        print(f'MapViewer needs the [viz] extra: pip install "pygeofetch[viz]" ({exc})')


## 9. Synthetic fallback — exercises the real clip + TimeSeriesAnalyzer pipeline

If live search/download wasn't available above, this builds 6 dates of
synthetic-but-realistic Landsat-style bands with a known, deliberately
injected linear vegetation decline in part of the AOI, writes them to
real georeferenced GeoTIFFs, and runs them through the **actual**
`Preprocessor.clip()` and `TimeSeriesAnalyzer` — not a separate
reimplementation — so this validates the real pipeline end-to-end even
without live credentials.


In [ ]:
if not have_real_data:
    print("Running synthetic fallback demonstration...")
    import rasterio
    from rasterio.crs import CRS
    from rasterio.transform import from_bounds

    np.random.seed(11)
    H, W = 120, 150
    full_bounds = (-1.85, 6.05, -1.55, 6.35)  # deliberately larger than the AOI, to prove clipping works
    transform = from_bounds(*full_bounds, W, H)
    crs = CRS.from_epsg(4326)

    y, x = np.mgrid[0:H, 0:W]
    # Center the injected decline on the AOI's REAL geographic centroid,
    # not an arbitrary full-scene pixel position — otherwise the decline
    # zone can miss the actual clipped region entirely (confirmed this
    # was happening: the AOI centroid falls at pixel col~76 in this grid,
    # not col~52 where an earlier version of this cell put it, causing
    # the "declining" signal to be clipped away before ever reaching the
    # time series analysis below).
    from rasterio.transform import rowcol
    _aoi_lons = [c[0] for c in obuasi_boundary_coords]
    _aoi_lats = [c[1] for c in obuasi_boundary_coords]
    _centroid_row, _centroid_col = rowcol(
        transform, sum(_aoi_lons) / len(_aoi_lons), sum(_aoi_lats) / len(_aoi_lats)
    )
    declining_zone = (
        (x - _centroid_col) ** 2 / (W * 0.12) ** 2
        + (y - _centroid_row) ** 2 / (H * 0.12) ** 2
    ) < 1

    synth_dates = ["2018-12-15", "2019-12-20", "2020-12-10", "2021-12-18", "2022-12-05", "2023-12-22"]
    date_bands = {}

    for i, date in enumerate(synth_dates):
        t = i / (len(synth_dates) - 1)
        # Realistic Landsat C2L2 DN counts, not arbitrary small numbers —
        # DN must be >= 7273 to produce a non-negative reflectance at all
        # given the official scale (0.0000275) and offset (-0.2); DNs of
        # 1100/3200 (an earlier version of this cell) scale to NEGATIVE
        # reflectance for both bands, producing a meaningless NDVI value
        # that doesn't behave like real vegetation data at all. These
        # values instead produce a realistic decline from NDVI ~0.67
        # (healthy forest) to ~0.30 (significant clearance) — a
        # magnitude consistent with real galamsey-driven deforestation.
        red = np.full((H, W), 9000, dtype=np.uint16)
        nir_base = 16000
        nir_decline_amount = t * 5500
        nir = np.where(declining_zone, int(nir_base - nir_decline_amount), nir_base).astype(np.uint16)

        date_dir = output_dir / date
        date_dir.mkdir(parents=True, exist_ok=True)
        red_path = date_dir / "SR_B4_raw.tif"
        nir_path = date_dir / "SR_B5_raw.tif"
        for path, arr in [(red_path, red), (nir_path, nir)]:
            with rasterio.open(path, "w", driver="GTiff", dtype="uint16", count=1,
                               width=W, height=H, crs=crs, transform=transform) as ds:
                ds.write(arr, 1)

        # Run through the REAL clip -> scale pipeline, same as Section 4
        red_clip = pp.clip(red_path, geometry=obuasi_geometry, output=str(date_dir / "red_clipped.tif"))
        nir_clip = pp.clip(nir_path, geometry=obuasi_geometry, output=str(date_dir / "nir_clipped.tif"))

        red_arr, profile = extractor.load_scaled_band(red_clip.output_path)
        nir_arr, _ = extractor.load_scaled_band(nir_clip.output_path)

        red_out = date_dir / "red_scaled_clipped.tif"
        nir_out = date_dir / "nir_scaled_clipped.tif"
        prof = profile.copy()
        prof.update(dtype="float32", count=1)
        for path, arr in [(red_out, red_arr), (nir_out, nir_arr)]:
            with rasterio.open(path, "w", **prof) as dst:
                dst.write(arr.astype(np.float32)[np.newaxis, :, :])

        date_bands[date] = {"RED": red_out, "NIR": nir_out}
        clip_profile = profile

    print(f"Built {len(date_bands)} synthetic dates, each clipped from a "
          f"{H}x{W} full scene down to the actual Obuasi boundary.\n")

    # Run the REAL TimeSeriesAnalyzer on this synthetic data
    stack = ts.build_index_stack(date_bands)
    print(stack)
    trend = ts.trend(stack)
    declining_trend = np.nanmean(trend[declining_zone[:trend.shape[0], :trend.shape[1]]] if trend.shape == declining_zone.shape else trend)
    print(f"\nMean trend: {np.nanmean(trend):.4f} NDVI/year")

    pl.plot_raster(
        trend, title="Obuasi — Synthetic NDVI Trend Demonstration",
        colormap="RdBu", vmin=-0.05, vmax=0.05, extent=aoi_extent,
        colorbar_label="NDVI / year", output=str(output_dir / "trend_map_synthetic.png"),
    )

    trend_classified = classify_trend(trend)
    pl.plot_classification(
        trend_classified, class_labels=TREND_CLASS_LABELS, class_colors=TREND_CLASS_COLORS,
        title="Obuasi — Synthetic Trend Classification", extent=aoi_extent,
        output=str(output_dir / "trend_classification_synthetic.png"),
    )

    declining_pct = 100 * np.nanmean(trend < -0.02)
    print(f"\nArea with clear declining trend: {declining_pct:.1f}%")
    print("This demonstrates the exact same clip-first + TimeSeriesAnalyzer")
    print("pipeline as Sections 3-7, run on synthetic data with a known,")
    print("deliberately-injected declining-vegetation signal.")
else:
    print("Real data was used above — synthetic fallback not needed.")


## 10. Save GIS-ready outputs


In [ ]:
from pygeofetch.processing.base import _safe_write_band

results_dir = output_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)

_safe_write_band(trend, clip_profile, results_dir / "ndvi_trend.tif")
_safe_write_band(trend_classified.astype(np.float32), clip_profile, results_dir / "trend_classification.tif")
if have_real_data:
    _safe_write_band(anomaly, clip_profile, results_dir / "ndvi_anomaly.tif")

print(f"Saved GIS-ready outputs to: {results_dir}")
for f in sorted(results_dir.glob("*.tif")):
    print(f"  {f.name}")


## Summary

**Why this is a stronger result than a two-date comparison:**

- 6 dates spanning 2018-2024, fit with an actual per-pixel linear trend
  — not a single subtraction that a single unlucky date pair could
  distort in either direction
- Clip-first pipeline — every scaling, masking, and index computation
  operated only on the actual Obuasi boundary, not the full downloaded
  scene
- Statistically-grounded anomaly detection (z-score vs. a baseline
  period), not just a raw difference map
- A district-wide time series trajectory you can point to directly:
  "here is what mean NDVI actually did across 6 years," not just two
  snapshots

**Reusing this for other locations:** the boundary polygon in Section 1
and the `date_windows` list in Section 2 are the only two things you
need to change — the entire clip-first + time-series pipeline is
location- and date-range-agnostic.

### References

- USGS. *Landsat 8-9 Collection 2 Level-2 Science Product Guide* (LSDS-1619).
- Berardino, P. et al. (2002). A new algorithm for surface deformation
  monitoring based on small baseline differential SAR interferograms.
  IEEE TGRS, 40(11), 2375-2383. (The SBAS least-squares approach this
  notebook's trend computation borrows the same statistical logic from,
  applied here to an optical index rather than InSAR phase.)
